In [7]:
import pandas as pd
import numpy as np
import warnings
from scipy.stats import spearmanr
import optuna
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

In [3]:
df = pd.read_parquet('BTC_1h_logRV_ML_dataset.parquet')
df.head()

,log_rv_lag_1h,log_rv_lag_2h,log_rv_lag_3h,log_rv_lag_6h,log_rv_lag_12h,log_rv_lag_24h,log_rv_lag_48h,log_rv_lag_72h,log_rv_lag_168h,log_rv_mean_3h,...,is_us_session,is_eu_us_overlap,is_low_vol_168h,is_high_vol_168h,log_rv_lag1_x_us_session,log_rv_lag1_x_weekend,log_volume_lag1_x_us_session,abs_ret_lag1_x_us_session,target_log_rv,target_rv
datetime,,,,,,,,,,,,,,,,,,,,,
2020-04-24 10:00:00+00:00,-11.037855,-10.484218,-10.835839,-10.888887,-10.680269,-11.994565,-12.063372,-10.541077,-11.278820,-10.785971,...,0,0,0,0,-0.000000,-0.0,0.000000,0.000000,-10.399756,0.000030
2020-04-24 11:00:00+00:00,-10.399756,-11.037855,-10.484218,-11.590795,-9.354094,-11.090701,-11.917092,-9.388464,-11.179881,-10.640610,...,0,0,0,0,-0.000000,-0.0,0.000000,0.000000,-9.136467,0.000108
2020-04-24 12:00:00+00:00,-9.136467,-10.399756,-11.037855,-12.173192,-10.142192,-10.344540,-11.376537,-10.698591,-11.013714,-10.191359,...,0,0,0,1,-0.000000,-0.0,0.000000,0.000000,-9.878886,0.000051
2020-04-24 13:00:00+00:00,-9.878886,-9.136467,-10.399756,-10.835839,-10.731020,-9.506733,-9.125376,-10.832082,-9.459211,-9.805036,...,1,1,0,0,-9.878886,-0.0,6.351896,0.001266,-11.220871,0.000013
2020-04-24 14:00:00+00:00,-11.220871,-9.878886,-9.136467,-10.484218,-11.638972,-6.899771,-9.461127,-10.254837,-10.610571,-10.078742,...,1,1,0,0,-11.220871,-0.0,5.967658,0.003403,-9.154555,0.000106


In [4]:
def make_walk_forward_splits(
    data,
    train_years=2,
    val_months=1,
    test_months=2,
    step_months=2,
    use_val=True
):
    start = data.index.min() + pd.DateOffset(years=train_years)
    end = data.index.max()

    splits = []
    split_start = start

    while True:
        train_start = split_start - pd.DateOffset(years=train_years)
        train_end = split_start

        if use_val:
            val_start = train_end
            val_end = val_start + pd.DateOffset(months=val_months)
            test_start = val_end
            test_end = test_start + pd.DateOffset(months=test_months)
        else:
            val_start = None
            val_end = None
            test_start = train_end
            test_end = test_start + pd.DateOffset(months=test_months)

        if test_end > end:
            break

        train_idx = (data.index >= train_start) & (data.index < train_end)
        test_idx = (data.index >= test_start) & (data.index < test_end)

        if use_val:
            val_idx = (data.index >= val_start) & (data.index < val_end)
        else:
            val_idx = None

        splits.append({
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "train_start": train_start,
            "train_end": train_end,
            "val_start": val_start,
            "val_end": val_end,
            "test_start": test_start,
            "test_end": test_end
        })

        split_start = split_start + pd.DateOffset(months=step_months)

    return splits

In [5]:
def qlike(y_true_rv, y_pred_rv, eps=1e-12):
    y_true_rv = np.maximum(y_true_rv, eps)
    y_pred_rv = np.maximum(y_pred_rv, eps)

    return np.mean(
        y_true_rv / y_pred_rv - np.log(y_true_rv / y_pred_rv) - 1
    )


def regression_metrics(y_true_log, y_pred_log, eps=1e-12):
    y_true_log = np.asarray(y_true_log)
    y_pred_log = np.asarray(y_pred_log)

    y_true_rv = np.exp(y_true_log) - eps
    y_pred_rv = np.exp(y_pred_log) - eps

    return {
        "mse_log": np.mean((y_true_log - y_pred_log) ** 2),
        "mae_log": np.mean(np.abs(y_true_log - y_pred_log)),
        "mse_rv": np.mean((y_true_rv - y_pred_rv) ** 2),
        "mae_rv": np.mean(np.abs(y_true_rv - y_pred_rv)),
        "qlike": qlike(y_true_rv, y_pred_rv),
        "pearson": np.corrcoef(y_true_log, y_pred_log)[0, 1],
        "spearman": spearmanr(y_true_log, y_pred_log).correlation
    }

In [6]:
def run_walk_forward_with_val(
    model_fn,
    dataset,
    feature_cols,
    target_col="target_log_rv",
    output_path="predictions_walk_forward_val.parquet"
):
    splits = make_walk_forward_splits(
        dataset,
        train_years=2,
        val_months=1,
        test_months=2,
        step_months=2,
        use_val=True
    )

    preds = []
    metrics = []

    for i, split in enumerate(splits):
        train = dataset.loc[split["train_idx"]]
        val = dataset.loc[split["val_idx"]]
        test = dataset.loc[split["test_idx"]]

        model = model_fn()

        model.fit(
            train[feature_cols],
            train[target_col],
            eval_set=[(val[feature_cols], val[target_col])]
        )

        y_pred = model.predict(test[feature_cols])
        y_true = test[target_col].values

        fold_preds = pd.DataFrame({
            "datetime": test.index,
            "fold": i,
            "y_true_log_rv": y_true,
            "y_pred_log_rv": y_pred,
            "y_true_rv": np.exp(y_true) - 1e-12,
            "y_pred_rv": np.exp(y_pred) - 1e-12
        })

        preds.append(fold_preds)

        fold_metrics = regression_metrics(y_true, y_pred)
        fold_metrics["fold"] = i
        fold_metrics["test_start"] = split["test_start"]
        fold_metrics["test_end"] = split["test_end"]
        metrics.append(fold_metrics)

    preds = pd.concat(preds).set_index("datetime")
    metrics = pd.DataFrame(metrics)

    preds.to_parquet(output_path)

    return preds, metrics, metrics.drop(columns=["fold", "test_start", "test_end"]).mean()

In [ ]:
class OptunaLinear:
    def __init__(self, model_type, n_trials=30):
        self.model_type = model_type
        self.n_trials = n_trials

    def fit(self, X, y, eval_set=None):
        X_val, y_val = eval_set[0]

        def objective(trial):
            alpha = trial.suggest_float("alpha", 1e-6, 10.0, log=True)
            model = Ridge(alpha=alpha) if self.model_type == "ridge" else Lasso(alpha=alpha, max_iter=20000)

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("model", model)
            ])

            pipe.fit(X, y)
            pred = pipe.predict(X_val)

            return mean_squared_error(y_val, pred)

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=self.n_trials, show_progress_bar=False)

        model = Ridge(alpha=study.best_params["alpha"]) if self.model_type == "ridge" else Lasso(alpha=study.best_params["alpha"], max_iter=20000)

        self.model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])

        self.model.fit(pd.concat([X, X_val]), pd.concat([y, y_val]))

        return self

    def predict(self, X):
        return self.model.predict(X)


df = pd.read_parquet("BTC_1h_logRV_ML_dataset.parquet").sort_index()

y = df["target_log_rv"]
rv = np.exp(y)

df["target_log_rv_1h"] = y.shift(-1)
df["target_log_rv_6h"] = np.log(rv.shift(-1).rolling(6).mean().shift(-5))
df["target_log_rv_24h"] = np.log(rv.shift(-1).rolling(24).mean().shift(-23))
df["target_log_rv_168h"] = np.log(rv.shift(-1).rolling(168).mean().shift(-167))

targets = [
    "target_log_rv_1h",
    "target_log_rv_6h",
    "target_log_rv_24h",
    "target_log_rv_168h",
]

feature_sets = {
    "HAR": [
        "log_rv_mean_6h",
        "log_rv_mean_24h",
        "log_rv_mean_168h",
        "log_rv_mean_720h",
    ],
    "HAR_24": [
        "log_rv_mean_24h",
        "log_rv_mean_72h",
        "log_rv_mean_168h",
    ],
    "HAR_Long": [
        "log_rv_mean_24h",
        "log_rv_mean_168h",
        "log_rv_mean_720h",
    ],
    "HAR_EWMA": [
        "log_rv_mean_24h",
        "log_rv_mean_168h",
        "log_rv_ewm_24h",
        "log_rv_ewm_168h",
    ],
    "HARX_Seasonal": [
        "log_rv_mean_24h",
        "log_rv_mean_168h",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "is_weekend",
        "is_us_session",
        "is_eu_us_overlap",
    ],
    "HARX_Liquidity": [
        "log_rv_mean_24h",
        "log_rv_mean_168h",
        "log_volume_mean_24h",
        "log_turnover_mean_24h",
        "log_dollar_volume_mean_24h",
    ],
    "HAR_Regime": [
        "log_rv_mean_24h",
        "log_rv_mean_168h",
        "log_rv_zscore_168h",
        "is_low_vol_168h",
        "is_high_vol_168h",
    ],
    "HARX_Full": [
        "log_rv_mean_6h",
        "log_rv_mean_24h",
        "log_rv_mean_168h",
        "log_rv_mean_720h",
        "log_rv_std_24h",
        "log_rv_std_168h",
        "log_rv_ewm_24h",
        "log_rv_ewm_168h",
        "log_rv_zscore_168h",
        "abs_ret_mean_24h",
        "abs_ret_mean_168h",
        "hl_range_mean_24h",
        "oc_range_mean_24h",
        "log_volume_mean_24h",
        "log_volume_mean_168h",
        "log_turnover_mean_24h",
        "log_dollar_volume_mean_24h",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "is_weekend",
        "is_us_session",
        "is_eu_us_overlap",
        "is_low_vol_168h",
        "is_high_vol_168h",
        "log_rv_lag1_x_us_session",
        "log_rv_lag1_x_weekend",
        "log_volume_lag1_x_us_session",
        "abs_ret_lag1_x_us_session",
    ],
}

all_preds = []
all_metrics = []

for target in targets:
    for spec, features in feature_sets.items():
        for model_type in ["ridge", "lasso"]:
            name = f"{model_type.upper()}_{spec}"

            data = df[features + [target]].replace([np.inf, -np.inf], np.nan).dropna()

            preds, metrics, avg_metrics = run_walk_forward_with_val(
                model_fn=lambda model_type=model_type: OptunaLinear(model_type, n_trials=30),
                dataset=data,
                feature_cols=features,
                target_col=target,
                output_path=f"{name}_{target}.parquet"
            )

            preds = preds.reset_index()
            preds["model"] = name
            preds["target"] = target
            all_preds.append(preds)

            metrics["model"] = name
            metrics["target"] = target
            all_metrics.append(metrics)

ridge_lasso_har_preds = pd.concat(all_preds)
ridge_lasso_har_metrics = pd.concat(all_metrics).reset_index(drop=True)

ridge_lasso_har_preds.to_parquet("ridge_lasso_har.parquet", index=False)
ridge_lasso_har_metrics.to_parquet("ridge_lasso_har_metrics.parquet", index=False)

[I 2026-05-08 23:47:20,842] A new study created in memory with name: no-name-3af3be6a-27db-4d7c-aac8-c2654cd9bc20
[I 2026-05-08 23:47:20,862] Trial 0 finished with value: 0.8081797508874302 and parameters: {'alpha': 1.3510184330101906}. Best is trial 0 with value: 0.8081797508874302.
[I 2026-05-08 23:47:20,865] Trial 1 finished with value: 0.8081728226005231 and parameters: {'alpha': 0.612158496681377}. Best is trial 1 with value: 0.8081728226005231.
[I 2026-05-08 23:47:20,868] Trial 2 finished with value: 0.8081670962215107 and parameters: {'alpha': 0.0014173436655707356}. Best is trial 2 with value: 0.8081670962215107.
[I 2026-05-08 23:47:20,870] Trial 3 finished with value: 0.8081670847360444 and parameters: {'alpha': 0.00019232062425798138}. Best is trial 3 with value: 0.8081670847360444.
[I 2026-05-08 23:47:20,872] Trial 4 finished with value: 0.808167085974271 and parameters: {'alpha': 0.00032438807655930214}. Best is trial 3 with value: 0.8081670847360444.
[I 2026-05-08 23:47:20